# Thème Numéro 2 - Facteurs Saisonniers et Succès au Speed Dating

## Question 3 - Les critères de sélection varient-ils selon la saison ?
L'importance accordée aux différents traits (attractivité, sincérité, intelligence, fun, ambition, intérêts communs) diffère-t-elle entre le printemps et l'automne ?

Pour chaque trait, on teste :
- **H0** : l'importance accordée à ce trait ne diffère pas entre Spring et Autumn (p > 0.05)
- **H1** : l'importance accordée à ce trait diffère entre Spring et Autumn (p ≤ 0.05)

**Variable dépendante — critères de sélection :**
Les participants avaient **100 points à répartir** entre 6 attributs avant l'événement (colonne `_1_1`), en fonction de l'importance qu'ils accordent à chaque critère chez un partenaire potentiel :
- `attr1_1` : Attractivité physique
- `sinc1_1` : Sincérité
- `intel1_1` : Intelligence
- `fun1_1` : Fun / Sens de l'humour
- `amb1_1` : Ambition
- `shar1_1` : Intérêts / hobbies en commun

Seuil de significativité : α = 0.05  

## 0. Chargement des données

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("Speed+Dating+Data.csv", encoding="MacRoman")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


## 1. Création des variables

In [3]:
SPRING_WAVES = [6, 7, 8, 9, 18, 19, 20, 21]

df['season'] = df['wave'].apply(lambda w: 'Spring' if w in SPRING_WAVES else 'Autumn')

TRAITS    = ['attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1']
LABELS    = ['Attractivité', 'Sincérité', 'Intelligence', 'Fun', 'Ambition', 'Intérêts communs']

# Un seul enregistrement par participant (les traits _1_1 sont constants sur toutes les lignes d'un même iid)
per_person = df.drop_duplicates(subset='iid')[['iid', 'wave', 'season'] + TRAITS].copy()

print(f"Nombre de participants : {len(per_person)}")
print(per_person['season'].value_counts().to_string())
per_person.head()

Nombre de participants : 551
season
Autumn    350
Spring    201


,iid,wave,season,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1
0,1,1,Autumn,15.0,20.0,20.0,15.0,15.0,15.0
10,2,1,Autumn,45.0,5.0,25.0,20.0,0.0,5.0
20,3,1,Autumn,35.0,10.0,35.0,10.0,10.0,0.0
30,4,1,Autumn,20.0,20.0,20.0,20.0,10.0,10.0
40,5,1,Autumn,20.0,5.0,25.0,25.0,10.0,15.0


## 2. Statistiques descriptives

In [5]:
descr = per_person.groupby('season')[TRAITS].mean().round(2)
descr.columns = LABELS
print("Points moyens alloués par trait et par saison (sur 100):\n ")
print(descr)

# vérification - la somme doit être ≈ 100
per_person['sum_traits'] = per_person[TRAITS].sum(axis=1)
print(f"\nSomme moyenne des traits (doit être ≈ 100):")
print(per_person.groupby('season')['sum_traits'].mean().round(2))

Points moyens alloués par trait et par saison (sur 100):
 
        Attractivité  Sincérité  Intelligence    Fun  Ambition  \
season                                                           
Autumn         24.47      17.16         20.52  17.25      9.96   
Spring         19.62      17.52         19.57  17.79     12.26   

        Intérêts communs  
season                    
Autumn             11.02  
Spring             13.24  

Somme moyenne des traits (doit être ≈ 100):
season
Autumn    98.48
Spring    99.43
Name: sum_traits, dtype: float64


## 3. Vérification de la normalité

Avant de réaliser un t-test indépendant, il faut vérifier la **normalité** pour chaque trait (test de Shapiro-Wilk sur chaque groupe)

In [9]:
print(f"{'Trait':<20} {'Shapiro Spring':>20} {'Shapiro Autumn':>20} {'Normalité'}")
print("-" * 75)

for trait, label in zip(TRAITS, LABELS):
    clean = per_person[['season', trait]].dropna()
    spring = clean[clean['season'] == 'Spring'][trait]
    autumn = clean[clean['season'] == 'Autumn'][trait]

    _, ps = stats.shapiro(spring)
    _, pa = stats.shapiro(autumn)
    normal = 'oui' if ps >= 0.05 and pa >= 0.05 else 'non (robuste n>30)'
    
    print(f"{label:<20} {ps:>20.4f} {pa:>20.4f} {normal}")


Trait                      Shapiro Spring       Shapiro Autumn Normalité
---------------------------------------------------------------------------
Attractivité                       0.0000               0.0000 non (robuste n>30)
Sincérité                          0.0000               0.0000 non (robuste n>30)
Intelligence                       0.0000               0.0000 non (robuste n>30)
Fun                                0.0000               0.0000 non (robuste n>30)
Ambition                           0.0000               0.0000 non (robuste n>30)
Intérêts communs                   0.0000               0.0000 non (robuste n>30)


## T-tests indépendants par trait
On rélise un t-test de Welch (equal_var=False) pour chaque trait. 
Seuil de significativité: α = 0.05.

In [15]:
results = []

print(f"{'Trait':<20} {'Spring':>8} {'Autumn':>8} {'t':>8} {'p':>10} {'Résultat'}")
print("-" * 80)

for trait, label in zip(TRAITS, LABELS):
    clean = per_person[['season', trait]].dropna()
    spring = clean[clean['season'] == 'Spring'][trait]
    autumn = clean[clean['season'] == 'Autumn'][trait]

    t, p = stats.ttest_ind(spring, autumn, equal_var=False)
    reject = p < 0.05
    resultat = 'H0 rejetée' if reject else 'H0 non rejetée'

    results.append({
        'trait': trait, 'label': label,
        'spring_mean': spring.mean(), 'autumn_mean': autumn.mean(),
        'n_spring': len(spring), 'n_autumn': len(autumn),
        't': t, 'p': p, 'sig': sig, 'reject': reject
    })

    print(f"{label:<20} {spring.mean():>8.2f} {autumn.mean():>8.2f} {t:>8.3f} {p:>10.4f} {resultat}")

results_df = pd.DataFrame(results)

Trait                  Spring   Autumn        t          p Résultat
--------------------------------------------------------------------------------
Attractivité            19.62    24.47   -4.801     0.0000 H0 rejetée
Sincérité               17.52    17.16    0.610     0.5421 H0 non rejetée
Intelligence            19.57    20.52   -1.786     0.0746 H0 non rejetée
Fun                     17.79    17.25    1.098     0.2728 H0 non rejetée
Ambition                12.26     9.96    4.359     0.0000 H0 rejetée
Intérêts communs        13.24    11.02    4.017     0.0001 H0 rejetée


## 5. Récapitulatif

In [16]:
print("Traits avec différence significative (α = 0.05) :")
sig_traits = results_df[results_df['reject']]
for _, row in sig_traits.iterrows():
    direction = 'Spring > Autumn' if row['spring_mean'] > row['autumn_mean'] else 'Autumn > Spring'
    print(f"  {row['label']:<20} : Spring={row['spring_mean']:.2f}, Autumn={row['autumn_mean']:.2f} → {direction} (p={row['p']:.4f})")

print("\nTraits sans différence significative :")
ns_traits = results_df[~results_df['reject']]
for _, row in ns_traits.iterrows():
    print(f"  {row['label']:<20} : Spring={row['spring_mean']:.2f}, Autumn={row['autumn_mean']:.2f} (p={row['p']:.4f})")

Traits avec différence significative (α = 0.05) :
  Attractivité         : Spring=19.62, Autumn=24.47 → Autumn > Spring (p=0.0000)
  Ambition             : Spring=12.26, Autumn=9.96 → Spring > Autumn (p=0.0000)
  Intérêts communs     : Spring=13.24, Autumn=11.02 → Spring > Autumn (p=0.0001)

Traits sans différence significative :
  Sincérité            : Spring=17.52, Autumn=17.16 (p=0.5421)
  Intelligence         : Spring=19.57, Autumn=20.52 (p=0.0746)
  Fun                  : Spring=17.79, Autumn=17.25 (p=0.2728)


## 6. Visualisation

In [17]:
# bar chart comparatif : moyennes Spring vs Autumn pour chaque trait
plot_data = pd.DataFrame({
    'Trait': LABELS * 2,
    'Saison': ['Spring'] * len(LABELS) + ['Autumn'] * len(LABELS),
    'Moyenne': [results_df[results_df['label']==l]['spring_mean'].values[0] for l in LABELS] +
               [results_df[results_df['label']==l]['autumn_mean'].values[0] for l in LABELS]
})

fig_bar = px.bar(
    plot_data,
    x='Trait',
    y='Moyenne',
    color='Saison',
    barmode='group',
    text=plot_data['Moyenne'].round(1),
    title="Importance moyenne accordée à chaque trait par saison (sur 100 points)",
    labels={'Moyenne': 'Points alloués (moyenne)', 'Trait': 'Critère'},
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_bar.update_traces(textposition='outside')
fig_bar.show()

c:\Users\PC\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [19]:
# box plots : un par trait
df_long = per_person.melt(
    id_vars=['iid', 'season'],
    value_vars=TRAITS,
    var_name='trait_code',
    value_name='points'
)
label_map = dict(zip(TRAITS, LABELS))
df_long['Trait'] = df_long['trait_code'].map(label_map)

fig_box = px.box(
    df_long.dropna(),
    x='Trait',
    y='points',
    color='season',
    title="Distribution des points alloués par trait et par saison",
    labels={'points': 'Points alloués', 'saison': 'Saison', 'Trait': 'Critère'},
    color_discrete_map={'Spring': '#2ecc71', 'Autumn': '#e67e22'}
)
fig_box.show()

## Conclusion


Contrairement aux questions Q1 et Q2, **la saison influence significativement l'importance accordée à plusieurs traits** (α = 0.05) :

| Trait | Spring | Autumn | Résultat |
|---|---|---|---|
| Attractivité | 19.62 | 24.47 | **Autumn > Spring *** |
| Sincérité | 17.52 | 17.16 | ns |
| Intelligence | 19.57 | 20.52 | ns |
| Fun | 17.79 | 17.25 | ns |
| Ambition | 12.26 | 9.96 | **Spring > Autumn *** |
| Intérêts communs | 13.24 | 11.02 | **Spring > Autumn *** |

**3 traits sur 6 présentent une différence significative.**

### Interprétation

Les participants de l'**automne** accordent davantage d'importance à l'**attracticité physique** (+4.85 points) - un trait immédiat et évaluable rapidement lors d'une rencontre courte.

Les participants du **printemps**, eux, valorisent davantage l'**ambition** (+2.3 points) et les **intérêts communs** (+2.22 points) - des traits plus profonds qui reflètent une comptabilité à long terme.

En revanche, la sincérité, l'intelligence et le fun sont valorisés de façon similaire dans les deux saisons, ce qui suggère que certains critères sont plus universels et stables que d'autres.

### Ce que ça nous dit
Les critères de sélection romantique ne sont pas universellement stables : le contexte (ici la cohorte/saison) peut influencer ce que les gens déclarent valoriser chez un partenaire potentiel, même si ce contexte n'affecte pas le succès final (Q1) ni le comportement de sélection (Q2).

### Limites à considérer

- La normalité n'est pas vérifiée pour aucun des traits (Shapiro-Wilk p < 0.001 dans tous les cas), mais le t-test de Welch reste robuste pour des échantillons de cette taille (n > 30).
- Les différences observées pourraient refléter des effets de cohorte plutôt que des effets saisonniers stricts — les mêmes individus ne participent pas en Spring et en Autumn
- Le groupe Spring reste le plus petit (n = 201 vs n = 350), ce qui limite la puissance statistique pour détecter un effet potentiellement faible.